In [14]:
!pip install hdbscan umap-learn seaborn scikit-learn matplotlib scipy pymatreader numpy
!pip install oasis-deconv


In [15]:
import os
import copy
import json
import random
import warnings
from collections import Counter, deque, defaultdict

import numpy as np
import pandas as pd
import joblib
import scipy.io as sio
from scipy import signal
from scipy.signal import welch, medfilt, resample
from scipy.stats import spearmanr
from pymatreader import read_mat

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from tqdm import tqdm

from sklearn.linear_model import RidgeCV
from sklearn.metrics import (classification_report, r2_score, accuracy_score, precision_recall_curve, auc,
                             normalized_mutual_info_score, adjusted_rand_score, confusion_matrix)
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, normalize, RobustScaler
from sklearn.metrics.pairwise import cosine_distances
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor

import hdbscan
import umap
from oasis.functions import deconvolve

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')

In [16]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f">>> 当前计算设备: {device}")


>>> 当前计算设备: cuda


In [17]:
DATA_BASE_PATH = '/content/drive/MyDrive/24hours_mPFC_multimodal_neural_activity_and_natural_behavior_dataset/Processed data/'
MOUSE_IDS = ['#1', '#2', '#3', '#4_1', '#4_2', '#5']
TEST_MOUSE_ID = '#5'

FS_CALCIUM = 1.92
SEQ_LEN_ENCODER = 5
TIME_LAG_SEC = 5.0
TIME_LAG_STEPS = int(TIME_LAG_SEC * FS_CALCIUM)

concept_names = ['EEG_Delta', 'EEG_Theta', 'Delta_Theta_Ratio', 'EMG_Power',
                 'Beh_Motor', 'Beh_Locomotion', 'Beh_Grooming', 'Beh_NestGrooming',
                 'Beh_NestActivity', 'Beh_Rearing', 'Beh_Drinking', 'Beh_Eating', 'Beh_Nesting']
stage_names = ['Wake', 'NREM', 'REM', 'Microarousal']


In [18]:
# =====================================================================
# Data Processing Pipeline
# =====================================================================
def get_parts_for_mouse(base_path, mouse_id):
    parts, part_idx = [], 1
    while True:
        part_path = os.path.join(base_path, mouse_id, f'part{part_idx}')
        if os.path.exists(part_path):
            parts.append(part_path)
            part_idx += 1
        else:
            break
    if not parts:
        fallback = os.path.join(base_path, mouse_id)
        if os.path.exists(fallback): parts = [fallback]
    return parts

def apply_causal_smoothing(X, window_size=5):
    kernel = np.ones(window_size) / window_size
    X_smoothed = np.zeros_like(X)
    for i in range(X.shape[1]):
        conv_res = np.convolve(X[:, i], kernel, mode='full')[:X.shape[0]]
        for j in range(window_size - 1):
            conv_res[j] = np.mean(X[:j+1, i])
        X_smoothed[:, i] = conv_res
    return X_smoothed

def create_sliding_windows(X, y, C, window_size):
    N = len(X)
    if N < window_size: return None, None, None
    shape_X = (N - window_size + 1, window_size, X.shape[1])
    strides_X = (X.strides[0], X.strides[0], X.strides[1])
    X_seq = np.lib.stride_tricks.as_strided(X, shape=shape_X, strides=strides_X)

    shape_C = (N - window_size + 1, window_size, C.shape[1])
    strides_C = (C.strides[0], C.strides[0], C.strides[1])
    C_seq = np.lib.stride_tricks.as_strided(C, shape=shape_C, strides=strides_C)
    return X_seq, y[window_size - 1:], C_seq

def load_data(part_path, handle_eeg_missing='truncate'):
    calcium_mat = sio.loadmat(os.path.join(part_path, 'Calcium imaging_Trace.mat'))
    dff, mask = calcium_mat['dff'], calcium_mat['mask'].flatten().astype(bool)

    beh_df = pd.read_csv(os.path.join(part_path, 'Behavior recording_Label.csv'))

    le = LabelEncoder()
    sleep_stage = le.fit_transform(beh_df['sleep_stage'])
    stage_names_local = le.classes_.tolist()

    beh_cols = ['motor','locomotion','grooming','nestgrooming','nestactivity','rearing','drinking','eating','nesting']
    beh_data = beh_df[beh_cols].values.astype(np.float32)

    eeg_file = os.path.join(part_path, 'EEGEMG recording_Filtered.mat')
    if os.path.exists(eeg_file):
        inner = read_mat(eeg_file)['filteredEEG']
        raw_eeg, raw_emg, fs, eeg_available = inner['EEG1'].flatten(), inner['EMG'].flatten(), 1000, True
    else:
        raw_eeg, raw_emg, fs, eeg_available = None, None, None, False

    valid_idx = mask
    X, y_sleep, y_beh = dff[:, valid_idx].T, sleep_stage[valid_idx], beh_data[valid_idx]

    if eeg_available:
        eeg_indices = np.linspace(0, len(raw_eeg) - 1, X.shape[0]).astype(int)
        last_valid_time_idx = np.searchsorted(eeg_indices, len(raw_eeg) - 1, side='right')
        if last_valid_time_idx < X.shape[0]:
            X, y_sleep, y_beh = X[:last_valid_time_idx], y_sleep[:last_valid_time_idx], y_beh[:last_valid_time_idx]
            eeg_indices = eeg_indices[:last_valid_time_idx]
    else:
        eeg_indices = None

    fs_original = 0.96 if os.path.basename(os.path.dirname(part_path)) == '#5' else 1.92
    if fs_original != FS_CALCIUM:
        ratio = FS_CALCIUM / fs_original
        new_len = int(X.shape[0] * ratio)
        X_resampled = np.zeros((new_len, X.shape[1]), dtype=X.dtype)
        for i in range(X.shape[1]): X_resampled[:, i] = signal.resample(X[:, i], new_len)
        X = X_resampled
        y_sleep = np.repeat(y_sleep, int(ratio))
        y_beh = np.repeat(y_beh, int(ratio), axis=0)
        if eeg_available:
            eeg_indices = np.clip(np.linspace(0, len(eeg_indices) - 1, new_len).astype(int), 0, len(raw_eeg) - 1)

    return X, y_sleep, y_beh, beh_cols, stage_names_local, raw_eeg, raw_emg, fs, eeg_indices

def extract_concept_features(raw_eeg, eeg_indices, beh_df, fs_eeg=1000, raw_emg=None):
    n_samples = len(beh_df)
    if raw_eeg is None or eeg_indices is None:
        delta_powers = theta_powers = emg_powers = np.zeros(n_samples, dtype=np.float32)
    else:
        half_window = int(1.0 * fs_eeg)
        delta_powers = np.full(n_samples, np.nan, dtype=np.float32)
        theta_powers = np.full(n_samples, np.nan, dtype=np.float32)
        emg_powers   = np.full(n_samples, np.nan, dtype=np.float32)

        for i, pos in enumerate(eeg_indices):
            start, end = pos - half_window, pos + half_window
            if start < 0 or end >= len(raw_eeg): continue

            segment_eeg = raw_eeg[start:end] - np.mean(raw_eeg[start:end])
            f, psd = welch(segment_eeg, fs=fs_eeg, nperseg=min(2 * half_window, 2048))
            delta_mask, theta_mask = (f >= 0.5) & (f <= 4.0), (f >= 4.0) & (f <= 8.0)
            delta_powers[i] = np.trapz(psd[delta_mask], f[delta_mask]) if np.any(delta_mask) else 0.0
            theta_powers[i] = np.trapz(psd[theta_mask], f[theta_mask]) if np.any(theta_mask) else 0.0

            if raw_emg is not None and end < len(raw_emg):
                emg_powers[i] = np.var(raw_emg[start:end])

        df_temp = pd.DataFrame({'delta': delta_powers, 'theta': theta_powers, 'emg': emg_powers}).fillna(method='ffill').fillna(0.0)
        delta_powers, theta_powers, emg_powers = df_temp['delta'].values, df_temp['theta'].values, df_temp['emg'].values

    ratio_dt = delta_powers / (theta_powers + 1e-8)
    behavior_features = np.column_stack([beh_df[col].values for col in ['motor','locomotion','grooming','nestgrooming','nestactivity','rearing','drinking','eating','nesting']])
    return np.column_stack([delta_powers, theta_powers, ratio_dt, emg_powers, behavior_features])

def build_future_dataset(parts, time_lag):
    C_seq_list, Y_fut_list, Y_curr_list = [], [], []
    for p in parts:
        if p.get('C_seq') is None: continue
        valid_len = len(p['C_seq']) - time_lag
        if valid_len > 0:
            C_seq_list.append(p['C_seq'][:valid_len])
            Y_curr_list.append(p['Y_seq'][:valid_len])
            Y_fut_list.append(p['Y_seq'][time_lag:])
    if not C_seq_list: return None, None, None
    return np.vstack(C_seq_list), np.concatenate(Y_fut_list), np.concatenate(Y_curr_list)


In [19]:
# =====================================================================
# Models
# =====================================================================
class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, pos_weight=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        return (self.alpha * (1 - pt) ** self.gamma * bce_loss).mean()

# TemporalBehaviorContrastive (Hybrid InfoNCE)
class TemporalBehaviorContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.1, alpha=0.5):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha

    def forward(self, out_seq, emb_final, labels):
        device = out_seq.device
        batch_size, seq_len, hidden_dim = out_seq.shape

        loss_time = torch.tensor(0.0, device=device)
        loss_beh = torch.tensor(0.0, device=device)

        # --- 1. Time-Contrastive Loss ---
        if self.alpha > 0 and seq_len > 1:
            z_t = F.normalize(out_seq[:, :-1, :].reshape(-1, hidden_dim), dim=1)
            z_t1 = F.normalize(out_seq[:, 1:, :].reshape(-1, hidden_dim), dim=1)

            sim_matrix_time = torch.matmul(z_t, z_t1.T) / self.temperature
            labels_time = torch.arange(z_t.size(0), device=device)
            loss_time = F.cross_entropy(sim_matrix_time, labels_time)

        # --- 2. Behavior-Contrastive Loss ---
        if self.alpha < 1.0:
            z_beh = F.normalize(emb_final, dim=1)
            sim_matrix_beh = torch.matmul(z_beh, z_beh.T) / self.temperature

            labels_beh = labels.contiguous().view(-1, 1)
            mask = torch.eq(labels_beh, labels_beh.T).float().to(device)

            logits_mask = torch.scatter(
                torch.ones_like(mask), 1,
                torch.arange(batch_size).view(-1, 1).to(device), 0
            )
            mask = mask * logits_mask

            logits_max, _ = torch.max(sim_matrix_beh, dim=1, keepdim=True)
            logits = sim_matrix_beh - logits_max.detach()

            exp_logits = torch.exp(logits) * logits_mask
            log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)

            mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-9)
            loss_beh = -mean_log_prob_pos.mean()

        return self.alpha * loss_time + (1 - self.alpha) * loss_beh


class DynamicsExtractor(nn.Module):
    def __init__(self, concept_dim, hidden_dim=32):
        super().__init__()
        # Delta, Variance, Energy
        self.fc = nn.Sequential(
            nn.Linear(concept_dim * 3, hidden_dim),
            nn.LayerNorm(hidden_dim), #  LayerNorm
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

    def forward(self, x):
        window = x[:, -5:, :]

        # 1. Δ Trend
        delta = x[:, -1, :] - window.mean(dim=1)

        # 2. Variance
        var = window.var(dim=1, unbiased=False)

        # 3. Energy
        energy = window.mean(dim=1)

        feat = torch.cat([delta, var, energy], dim=-1)
        return self.fc(feat)

class DualMechanismNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_classes=4, future_steps=5):
        super().__init__()
        self.future_steps = future_steps
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.2)

        # --- State ---
        self.fc_states = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Linear(32, num_classes))
            for _ in range(future_steps)
        ])

        # --- Transition ---
        self.dynamics = DynamicsExtractor(input_dim, hidden_dim=32)
        self.fc_trans = nn.Sequential(
            nn.Linear(hidden_dim + 32, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x, return_emb=False):
        # 1. State
        out_seq, (hn, cn) = self.lstm(x)
        emb_final = hn[-1]

        logits_s = torch.stack([head(emb_final) for head in self.fc_states], dim=1)

        # 2. Transition
        dyn_feat = self.dynamics(x)

        trans_input = torch.cat([emb_final.detach(), dyn_feat], dim=-1)
        logits_t = self.fc_trans(trans_input)

        if return_emb:
            return logits_s, logits_t, emb_final, out_seq
        return logits_s, logits_t

def supcon_loss(features, labels, temperature=0.1):
    """
    Supervised Contrastive Loss
    """
    device = features.device
    features = F.normalize(features, p=2, dim=1)
    batch_size = features.shape[0]

    labels = labels.contiguous().view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(device)

    logits_mask = torch.scatter(
        torch.ones_like(mask), 1,
        torch.arange(batch_size).view(-1, 1).to(device), 0
    )
    mask = mask * logits_mask

    anchor_dot_contrast = torch.div(torch.matmul(features, features.T), temperature)

    logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
    logits = anchor_dot_contrast - logits_max.detach()

    exp_logits = torch.exp(logits) * logits_mask
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)

    mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-9)
    loss = -mean_log_prob_pos.mean()

    return loss

class PrototypeRegistry:
    """
    Moving Average Prototypes
    """
    def __init__(self, emb_dim=64, num_classes=4, momentum=0.9, device='cuda'):
        self.prototypes = torch.zeros(num_classes, emb_dim, device=device)
        self.momentum = momentum
        self.is_initialized = torch.zeros(num_classes, dtype=torch.bool, device=device)
        self.num_classes = num_classes

    def update(self, features, labels):
        features = features.detach()
        for c in range(self.num_classes):
            mask = (labels == c)
            if mask.any():
                class_feat = features[mask].mean(dim=0)
                if not self.is_initialized[c]:
                    self.prototypes[c] = class_feat
                    self.is_initialized[c] = True
                else:
                    self.prototypes[c] = self.momentum * self.prototypes[c] + (1 - self.momentum) * class_feat
        self.prototypes.data = F.normalize(self.prototypes.data, p=2, dim=1)

    def get_prototypes(self):
        return self.prototypes


In [20]:
# =====================================================================
# Evaluation
# =====================================================================
def moving_avg_probs(probs, k=3):
    kernel = np.ones(k) / k
    if probs.ndim == 1:
        return np.convolve(probs, kernel, mode='same')
    else:
        smoothed = np.zeros_like(probs)
        for c in range(probs.shape[1]):
            smoothed[:, c] = np.convolve(probs[:, c], kernel, mode='same')
        return smoothed

def calculate_ece(probs, labels, n_bins=10):
    if hasattr(probs, 'detach'):
        probs = probs.detach().cpu().numpy()
    if hasattr(labels, 'detach'):
        labels = labels.detach().cpu().numpy()

    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = predictions == labels

    ece = 0.0
    bin_boundaries = np.linspace(0, 1, n_bins + 1)

    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(accuracies[in_bin])
            avg_confidence_in_bin = np.mean(confidences[in_bin])
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin

    return ece

def temporal_filter(dynamics_score, threshold, k=3):
    mask = (dynamics_score > threshold).float()

    if mask.dim() == 1:
        mask_3d = mask.unsqueeze(0).unsqueeze(0)
    elif mask.dim() == 2:
        mask_3d = mask.unsqueeze(1)

    kernel = torch.ones(1, 1, k, device=dynamics_score.device)
    conv = F.conv1d(mask_3d, kernel, padding=k-1)

    trigger = (conv[..., :mask_3d.shape[-1]] >= (k - 1e-5)).float()
    trigger[..., :k-1] = 0

    return trigger.view_as(dynamics_score)


def apply_cooldown(trigger_1d, cooldown=5):
    assert trigger_1d.ndim == 1, "1D"
    if isinstance(trigger_1d, torch.Tensor):
        final = trigger_1d.clone()
    else:
        final = trigger_1d.copy()

    last_idx = -cooldown
    for t in range(len(trigger_1d)):
        if trigger_1d[t] == 1:
            if (t - last_idx) < cooldown:
                final[t] = 0
            else:
                last_idx = t

    return final

def calculate_transition_metrics(y_true_trans, y_pred_trans, y_prob_trans, pre_frames=10, post_frames=3, fs=1.92):
    precision_curve, recall_curve, _ = precision_recall_curve(y_true_trans, y_prob_trans)
    auprc = auc(recall_curve, precision_curve)

    true_trans_indices = np.where(y_true_trans == 1)[0]
    pred_trans_indices = np.where(y_pred_trans == 1)[0]

    hits = 0
    lead_times = []
    valid_alarms = set()

    for idx in true_trans_indices:
        start_idx = max(0, idx - pre_frames)
        end_idx = min(len(y_pred_trans), idx + post_frames + 1)

        window_preds = y_pred_trans[start_idx : end_idx]

        if np.any(window_preds == 1):
            hits += 1
            alarm_indices_in_window = np.where(window_preds == 1)[0]
            first_alarm_idx_absolute = start_idx + alarm_indices_in_window[0]

            lead_time_sec = (idx - first_alarm_idx_absolute) / fs
            if lead_time_sec >= 0:
                lead_times.append(lead_time_sec)

            for a_idx in alarm_indices_in_window:
                valid_alarms.add(start_idx + a_idx)

    event_recall = hits / len(true_trans_indices) if len(true_trans_indices) > 0 else 0

    total_alarms = len(pred_trans_indices)
    valid_alarm_count = len(valid_alarms)
    precision_val = valid_alarm_count / total_alarms if total_alarms > 0 else 0

    avg_lead_time = np.mean(lead_times) if len(lead_times) > 0 else 0

    return auprc, precision_val, event_recall, avg_lead_time

def compute_transition_recall(y_true_fut, y_pred, transition_mask):
    mask = np.asarray(transition_mask, dtype=bool)
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.asarray(y_true_fut)[mask] == np.asarray(y_pred)[mask])

def build_transition_prior(y_curr_train, y_fut_train):
    """
     P(y_fut | y_curr)
    """
    counts = defaultdict(lambda: defaultdict(int))
    y_curr_train = np.asarray(y_curr_train)
    y_fut_train = np.asarray(y_fut_train)

    for cur, fut in zip(y_curr_train, y_fut_train):
        counts[cur][fut] += 1

    probs = {}
    for cur, d in counts.items():
        total = sum(d.values())
        probs[cur] = {k: v / total for k, v in d.items()}
    return probs

def predict_transition_prior(y_curr, transition_probs):
    y_curr = np.asarray(y_curr)
    preds = []
    for cur in y_curr:
        d = transition_probs.get(cur, None)
        if not d:
            preds.append(cur)
        else:
            preds.append(max(d.items(), key=lambda kv: kv[1])[0])
    return np.asarray(preds)

def majority_baseline(y_train_fut, size):
    values, counts = np.unique(np.asarray(y_train_fut), return_counts=True)
    majority = values[np.argmax(counts)]
    return np.full(size, majority)

def temporal_smoothing_baseline(y_curr, window=10):
    y_curr = np.asarray(y_curr)
    y_pred = []
    for i in range(len(y_curr)):
        window_vals = y_curr[max(0, i - window):i + 1]
        values, counts = np.unique(window_vals, return_counts=True)
        y_pred.append(values[np.argmax(counts)])
    return np.asarray(y_pred)

def block_permutation_test(y_true_fut, y_pred, transition_mask, block_size=100, n_perm=500, seed=42):
    rng = np.random.default_rng(seed)
    y_true_fut = np.asarray(y_true_fut)
    y_pred = np.asarray(y_pred)
    transition_mask = np.asarray(transition_mask, dtype=bool)

    n = len(y_pred)
    blocks = [np.arange(i, min(i + block_size, n)) for i in range(0, n, block_size)]

    real_score = compute_transition_recall(y_true_fut, y_pred, transition_mask)
    perm_scores = []

    for _ in range(n_perm):
        order = rng.permutation(len(blocks))
        perm_idx = np.concatenate([blocks[j] for j in order])

        y_pred_perm = y_pred[perm_idx]

        perm_scores.append(compute_transition_recall(y_true_fut, y_pred_perm, transition_mask))

    perm_scores = np.asarray(perm_scores)
    p_value = np.mean(perm_scores >= real_score)
    z_score = (real_score - perm_scores.mean()) / (perm_scores.std() + 1e-8)

    return {"real": real_score, "p": p_value, "z": z_score}

def bootstrap_ci(y_true_fut, y_pred, transition_mask, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    indices = np.where(np.asarray(transition_mask, dtype=bool))[0]
    scores = []
    for _ in range(n_boot):
        sample_idx = rng.choice(indices, size=len(indices), replace=True)
        scores.append(np.mean(np.asarray(y_true_fut)[sample_idx] == np.asarray(y_pred)[sample_idx]))
    scores = np.asarray(scores)
    return {
        "mean": float(scores.mean()),
        "ci_lower": float(np.percentile(scores, 2.5)),
        "ci_upper": float(np.percentile(scores, 97.5)),
    }

def full_evaluation(y_train_curr, y_train_fut, y_test_curr, y_test_fut, y_pred_model, transition_mask):
    results = {}

    results["model"] = compute_transition_recall(y_test_fut, y_pred_model, transition_mask)

    trans_prob = build_transition_prior(y_train_curr, y_train_fut)
    results["baseline_transition"] = compute_transition_recall(
        y_test_fut,
        predict_transition_prior(y_test_curr, trans_prob),
        transition_mask
    )

    results["baseline_majority"] = compute_transition_recall(
        y_test_fut,
        majority_baseline(y_train_fut, len(y_test_fut)),
        transition_mask
    )

    results["baseline_temporal"] = compute_transition_recall(
        y_test_fut,
        temporal_smoothing_baseline(y_test_curr),
        transition_mask
    )

    perm = block_permutation_test(y_test_fut, y_pred_model, transition_mask)
    ci = bootstrap_ci(y_test_fut, y_pred_model, transition_mask)

    return {"scores": results, "permutation": perm, "bootstrap": ci}

In [21]:
# =====================================================================
# Metacognition & Calibration
# =====================================================================
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits): return logits / self.temperature

    def fit(self, logits, labels, lr=0.01, max_iter=50):
        self.to(logits.device)
        nll_criterion = nn.CrossEntropyLoss()
        optimizer = optim.LBFGS([self.temperature], lr=lr, max_iter=max_iter)
        def eval():
            optimizer.zero_grad()
            loss = nll_criterion(self.forward(logits), labels)
            loss.backward()
            return loss
        optimizer.step(eval)
        return self.temperature.item()

def build_concept_anchors(model, C_tensor, Y_state_tensor, Y_trans_tensor, num_classes=4, device='cuda'):
    model.train()
    anchor_dict_state = {i: [] for i in range(num_classes)}

    dataset = TensorDataset(torch.FloatTensor(C_tensor).to(device),
                            torch.LongTensor(Y_state_tensor).to(device),
                            torch.FloatTensor(Y_trans_tensor).to(device))
    loader = DataLoader(dataset, batch_size=512, shuffle=True)

    for X_batch, Y_s_batch, _ in loader:
        X_batch.requires_grad_(True)
        logits_s_all, _, _, _ = model(X_batch, return_emb=True)
        logits_s = logits_s_all[:, -1, :]

        for i in range(len(X_batch)):
            label_s = Y_s_batch[i].item()
            grad_s = torch.autograd.grad(logits_s[i, label_s], X_batch, retain_graph=True)[0]
            anchor_dict_state[label_s].append(grad_s[i].abs().mean(dim=0).detach())

    final_anchors_state = {}
    for c in range(num_classes):
        if len(anchor_dict_state[c]) > 0:
            avg = torch.stack(anchor_dict_state[c]).mean(dim=0)
            final_anchors_state[c] = avg / (avg.max() + 1e-9)

    model.eval()
    return final_anchors_state

class CalibratedMetacognitiveLoop(nn.Module):
    def __init__(self, base_model, anchor_dict_state, prototypes,
                 temperature=1.0, entropy_s_thresh=1.0, proto_dist_thresh=0.4,
                 mask_lr=1.0, max_iters=3):
        super().__init__()
        self.model = base_model
        self.anchor_dict_state = anchor_dict_state
        self.prototypes = prototypes
        self.temperature = temperature

        self.entropy_s_thresh = entropy_s_thresh
        self.proto_dist_thresh = proto_dist_thresh
        self.mask_lr = mask_lr
        self.max_iters = max_iters

    def forward(self, x_seq, apply_intervention=True):
        device = x_seq.device
        batch_size, seq_len, num_concepts = x_seq.shape

        self.model.eval()
        with torch.no_grad():
            logits_s_all, logits_t, emb, _ = self.model(x_seq, return_emb=True)
            logits_s = logits_s_all[:, -1, :]

            probs_s = F.softmax(logits_s / self.temperature, dim=-1)
            entropy_s = -torch.sum(probs_s * torch.log(probs_s + 1e-9), dim=-1)
            probs_t = torch.sigmoid(logits_t).squeeze(-1)

            emb_norm = F.normalize(emb, p=2, dim=1)
            cos_sim = torch.matmul(emb_norm, self.prototypes.T)
            proto_dist = 1.0 - torch.max(cos_sim, dim=1)[0]

            delta_c = x_seq[:, -1, :] - x_seq[:, -5:, :].mean(dim=1)
            dynamics_score = torch.norm(delta_c, p=2, dim=-1)

        trigger_s = (entropy_s > self.entropy_s_thresh) | (proto_dist > self.proto_dist_thresh)

        if not apply_intervention or not trigger_s.any():
            return probs_s, probs_t, dynamics_score, None

        raw_mask_s = torch.zeros((batch_size, num_concepts), device=device)

        idx = torch.where(trigger_s)[0]
        x_triggered = x_seq[idx]

        self.model.train()
        with torch.enable_grad():
            raw_mask_sub = torch.zeros((len(idx), num_concepts), device=device)

            for iteration in range(self.max_iters):
                current_mask = torch.sigmoid(raw_mask_sub) * 2.0
                masked_x = x_triggered * current_mask.unsqueeze(1)
                masked_x.requires_grad_(True)

                logits_sub_all, _, _, _ = self.model(masked_x, return_emb=True)
                logits_sub = logits_sub_all[:, -1, :]
                tentative_classes = torch.argmax(F.softmax(logits_sub, dim=-1), dim=-1)

                ig_steps = 20
                baseline = torch.zeros_like(masked_x)
                alphas = torch.linspace(0, 1, steps=ig_steps, device=device).reshape(-1, 1, 1, 1)
                scaled_inputs = baseline.unsqueeze(0) + alphas * (masked_x.unsqueeze(0) - baseline.unsqueeze(0))
                scaled_inputs = scaled_inputs.reshape(ig_steps * len(idx), seq_len, num_concepts)
                scaled_inputs.requires_grad_(True)

                logits_ig_all, _, _, _ = self.model(scaled_inputs, return_emb=True)
                logits_ig = logits_ig_all[:, -1, :]

                target_classes_ig = tentative_classes.repeat(ig_steps)
                indices_ig = torch.arange(ig_steps * len(idx))
                scores_ig = logits_ig[indices_ig, target_classes_ig].sum()
                grads_ig = torch.autograd.grad(scores_ig, scaled_inputs)[0]

                rt_attribution = ((masked_x - baseline) * grads_ig.reshape(ig_steps, len(idx), seq_len, num_concepts).mean(dim=0)).abs().mean(dim=1)
                rt_attribution = rt_attribution / (rt_attribution.max(dim=1, keepdim=True)[0] + 1e-9)

                anchor_batch = torch.stack([self.anchor_dict_state[c.item()] for c in tentative_classes])
                divergence = F.relu(rt_attribution - anchor_batch)

                weight = (entropy_s[idx] / 1.386).unsqueeze(1)
                penalty = divergence * weight

                raw_mask_sub = raw_mask_sub - self.mask_lr * penalty

            raw_mask_s[idx] = raw_mask_sub

        self.model.eval()
        with torch.no_grad():
            final_mask_s = torch.sigmoid(raw_mask_s) * 2.0
            logits_s_final_all, _, _, _ = self.model(x_seq * final_mask_s.unsqueeze(1), return_emb=True)
            final_probs_s = F.softmax(logits_s_final_all[:, -1, :] / self.temperature, dim=-1)

            final_probs_t = probs_t

        return final_probs_s, final_probs_t, dynamics_score, final_mask_s

In [22]:
# =====================================================================
# Execution Stage 1 & 2
# =====================================================================
print("\n>>> [1] Start building a multimodal time series dataset...")
all_valid_parts = []
for mouse_id in MOUSE_IDS:
    for path in get_parts_for_mouse(DATA_BASE_PATH, mouse_id):
        try:
            X, y_sleep, y_beh, beh_cols, st_names, raw_eeg, raw_emg, fs, eeg_indices = load_data(path)
            X = apply_causal_smoothing(X, window_size=5)
            C_raw = extract_concept_features(raw_eeg, eeg_indices, pd.DataFrame(y_beh, columns=beh_cols), fs if fs else 1000, raw_emg)
            if np.isnan(C_raw).sum() > 0.1 * C_raw.size or np.any(np.nanstd(C_raw, axis=0) == 0): continue
            all_valid_parts.append({'mouse_id': mouse_id, 'X': X, 'Y': y_sleep, 'C_raw': C_raw})
        except Exception as e:
            pass

train_val_parts, test_parts = [p for p in all_valid_parts if p['mouse_id'] != TEST_MOUSE_ID], [p for p in all_valid_parts if p['mouse_id'] == TEST_MOUSE_ID]
scaler_C = RobustScaler().fit(np.vstack([p['C_raw'] for p in train_val_parts]))
for p in all_valid_parts: p['C_norm'] = scaler_C.transform(p['C_raw'])

print("\n>>> [2] CBM concept screening and dataset locking...")
concept_r2_scores = {name: [] for name in concept_names}
for p in train_val_parts:
    encoder = MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=200).fit(p['X'], p['C_norm'])
    for i, name in enumerate(concept_names): concept_r2_scores[name].append(r2_score(p['C_norm'][:, i], encoder.predict(p['X'])[:, i]))

good_concepts = [name for name in concept_names if np.mean(concept_r2_scores[name]) > 0.1]
good_indices = [concept_names.index(name) for name in good_concepts]
concept_names = good_concepts
print(f"-> Keeping valid concepts ({len(concept_names)}): {concept_names}")

for p in all_valid_parts:
    p['C_norm'] = p['C_norm'][:, good_indices]
    p['X_seq'], p['Y_seq'], p['C_seq'] = create_sliding_windows(p['X'], p['Y'], p['C_norm'], SEQ_LEN_ENCODER)

C_train_seq, Y_train_fut, Y_train_curr = build_future_dataset(train_val_parts, TIME_LAG_STEPS)
C_test_seq, Y_test_fut, Y_test_curr = build_future_dataset(test_parts, TIME_LAG_STEPS)

# Create multi-task switching identifier
Y_train_is_trans = (Y_train_curr != Y_train_fut).astype(np.float32)
Y_test_is_trans = (Y_test_curr != Y_test_fut).astype(np.float32)
transition_indices = np.where(Y_test_is_trans == 1)[0]
print(f"-> Training set number of sequences: {C_train_seq.shape[0]} | Test set number of sequences: {C_test_seq.shape[0]}")


>>> [1] Start building a multimodal time series dataset...

>>> [2] CBM concept screening and dataset locking...
-> Keeping valid concepts (11): ['EEG_Delta', 'EEG_Theta', 'Delta_Theta_Ratio', 'EMG_Power', 'Beh_Motor', 'Beh_Locomotion', 'Beh_Grooming', 'Beh_NestGrooming', 'Beh_NestActivity', 'Beh_Eating', 'Beh_Nesting']
-> Training set number of sequences: 673786 | Test set number of sequences: 171559


In [23]:
# =====================================================================
# Execution Stage 3
# =====================================================================
print("\n>>> [3] Training multi-head multi-task prediction network (Driver: TemporalBehaviorContrastive hybrid manifold)...")

future_steps = 5
Y_train_fut_seq = np.tile(Y_train_fut[:, np.newaxis], (1, future_steps))

true_pos_weight = torch.tensor([((Y_train_is_trans == 0).sum() / ((Y_train_is_trans == 1).sum() + 1e-5)).item()], dtype=torch.float32, device=device)

criterion_state = nn.CrossEntropyLoss()
criterion_trans = BinaryFocalLoss(alpha=0.25, gamma=2.0, pos_weight=true_pos_weight)

criterion_tbc = TemporalBehaviorContrastiveLoss(temperature=0.1, alpha=0.5).to(device)
proto_registry = PrototypeRegistry(emb_dim=64, num_classes=4, momentum=0.9, device=device)

mt_model = DualMechanismNet(input_dim=C_train_seq.shape[2], future_steps=future_steps).to(device)
optimizer_mt = optim.AdamW(mt_model.parameters(), lr=1e-3, weight_decay=1e-4)

dataset_mt = TensorDataset(
    torch.FloatTensor(C_train_seq).to(device),
    torch.LongTensor(Y_train_fut_seq).to(device),
    torch.FloatTensor(Y_train_is_trans).to(device)
)
loader_mt = DataLoader(dataset_mt, batch_size=1024, shuffle=True)

mt_model.train()
for epoch in range(12):
    total_loss, total_loss_s, total_loss_t, total_loss_contrastive = 0, 0, 0, 0
    for bx, by_state_seq, by_trans in loader_mt:
        optimizer_mt.zero_grad()

        logits_s, logits_t, emb_final, out_seq = mt_model(bx, return_emb=True)

        loss_s = 0
        for step in range(future_steps):
            loss_s += criterion_state(logits_s[:, step, :], by_state_seq[:, step])
        loss_s = loss_s / future_steps

        loss_t = criterion_trans(logits_t.squeeze(), by_trans)

        loss_tbc = criterion_tbc(out_seq, emb_final, by_state_seq[:, -1])

        loss = loss_s + 5.0 * loss_t + 0.1 * loss_tbc
        loss.backward()
        optimizer_mt.step()

        proto_registry.update(emb_final, by_state_seq[:, -1])

        total_loss += loss.item()
        total_loss_s += loss_s.item()
        total_loss_t += loss_t.item()
        total_loss_contrastive += loss_tbc.item()

    if (epoch+1) % 4 == 0:
        batches = len(loader_mt)
        print(f"  -> Epoch [{epoch+1}/12] Total: {total_loss/batches:.4f} | State: {total_loss_s/batches:.4f} | Focal_T: {total_loss_t/batches:.4f} | TemporalBehaviorContrastive: {total_loss_contrastive/batches:.4f}")


>>> [3] Training multi-head multi-task prediction network (Driver: TemporalBehaviorContrastive hybrid manifold)...
  -> Epoch [4/12] Total: 1.5268 | State: 0.3089 | Focal_T: 0.1596 | TemporalBehaviorContrastive: 4.2013
  -> Epoch [8/12] Total: 1.5082 | State: 0.3049 | Focal_T: 0.1590 | TemporalBehaviorContrastive: 4.0831
  -> Epoch [12/12] Total: 1.5009 | State: 0.3037 | Focal_T: 0.1586 | TemporalBehaviorContrastive: 4.0398


In [24]:
# =====================================================================
# Execution Stage 4
# =====================================================================
print("\n>>> [4] Transition Warning Statistical Evaluation (5s) ")

X_test_tensor = torch.FloatTensor(C_test_seq).to(device)

mt_model.eval()
with torch.no_grad():
    logits_s, _ = mt_model(X_test_tensor)
    pred_future_mt = torch.argmax(logits_s[:, -1, :], dim=-1).cpu().numpy()

transition_mask = np.zeros(len(Y_test_fut), dtype=bool)
transition_mask[transition_indices] = True

eval_results = full_evaluation(
    y_train_curr=Y_train_curr,
    y_train_fut=Y_train_fut,
    y_test_curr=Y_test_curr,
    y_test_fut=Y_test_fut,
    y_pred_model=pred_future_mt,
    transition_mask=transition_mask
)

model_recall = eval_results['scores']['model'] * 100
best_baseline = max(eval_results['scores']['baseline_transition'],
                    eval_results['scores']['baseline_temporal'],
                    eval_results['scores']['baseline_majority']) * 100

print(f"  · Model Recall (DualMechanismNet) : {model_recall:.2f}%")
print(f"  · Baseline (Prior Transition): {eval_results['scores']['baseline_transition'] * 100:.2f}%")
print(f"  · Baseline (Temporal Inertia): {eval_results['scores']['baseline_temporal'] * 100:.2f}%")
print(f"  · Baseline (Majority Class)  : {eval_results['scores']['baseline_majority'] * 100:.2f}%")
print(f"  · Δ (vs Best Baseline)     : +{(model_recall - best_baseline):.2f}%")
print(f"  · Bootstrap 95% CI         : [{eval_results['bootstrap']['ci_lower']*100:.2f}%, {eval_results['bootstrap']['ci_upper']*100:.2f}%]")
print(f"  · Block Permutation        : p = {eval_results['permutation']['p']:.4f}, Z = {eval_results['permutation']['z']:.2f}")



>>> [4] Transition Warning Statistical Evaluation (5s) 
  · Model Recall (DualMechanismNet) : 25.96%
  · Baseline (Prior Transition): 0.00%
  · Baseline (Temporal Inertia): 3.14%
  · Baseline (Majority Class)  : 5.22%
  · Δ (vs Best Baseline)     : +20.74%
  · Bootstrap 95% CI         : [24.87%, 27.05%]
  · Block Permutation        : p = 0.0000, Z = 8.82


In [25]:
# =====================================================================
# Execution Stage 5
# =====================================================================
print("\n>>> [5] Confidence calibration and metacognitive module initialization...")

X_test_tensor = torch.FloatTensor(C_test_seq).to(device)
test_Y_tensor = torch.LongTensor(Y_test_fut).to(device)
test_Y_trans_tensor = torch.FloatTensor(Y_test_is_trans).to(device)

all_logits, all_labels = [], []
with torch.no_grad():
    for i in range(0, len(X_test_tensor), 512):
        logits_s, _, _, _ = mt_model(X_test_tensor[i:i+512], return_emb=True)
        all_logits.append(logits_s[:, -1, :].detach())
        all_labels.append(test_Y_tensor[i:i+512].detach())

gathered_logits, gathered_labels = torch.cat(all_logits, dim=0), torch.cat(all_labels, dim=0)

scaler = TemperatureScaler()
optimal_T_raw = scaler.fit(gathered_logits, gathered_labels)

probs_initial = F.softmax(gathered_logits, dim=1)
probs_calibrated = F.softmax(gathered_logits / optimal_T_raw, dim=1)

initial_ece = calculate_ece(probs_initial, gathered_labels)
final_ece = calculate_ece(probs_calibrated, gathered_labels)

if final_ece > initial_ece:
    print(f"  -> Revert triggered: native ECE ({initial_ece:.4f}) is already excellent. Optimizing NLL degrades ECE to {final_ece:.4f}, rejecting scaling factor.")
    optimal_T = 1.0
    print(f"  -> ECE maintained: {initial_ece:.4f} (T=1.0000)")
else:
    optimal_T = optimal_T_raw
    print(f"  -> ECE improved: {initial_ece:.4f} → {final_ece:.4f} (T={optimal_T:.4f})")

new_anchor_dict_state = build_concept_anchors(
    model=mt_model,
    C_tensor=C_train_seq,
    Y_state_tensor=Y_train_fut,
    Y_trans_tensor=Y_train_is_trans,
    device=device
)


meta_loop_model = CalibratedMetacognitiveLoop(
    base_model=mt_model,
    anchor_dict_state=new_anchor_dict_state,
    prototypes=proto_registry.get_prototypes(),
    temperature=optimal_T,
    entropy_s_thresh=1.0,
    proto_dist_thresh=0.4,
    mask_lr=1.0,
    max_iters=3
).to(device)


>>> [5] Confidence calibration and metacognitive module initialization...
  -> ECE improved: 0.1196 → 0.1012 (T=1.6129)


In [26]:
# =====================================================================
# Execution Stage 6
# =====================================================================
print("\n>>> [6] Dynamics-Driven + Temporal Filter...")

sys1_probs_s_all, sys2_probs_s_all, sys1_probs_t_all = [], [], []
dynamics_scores_all, sys2_triggers_all = [], []

meta_loop_model.eval()
for i in tqdm(range(0, len(X_test_tensor), 256), desc="Dual-system inference"):
    bx = X_test_tensor[i:i+256]
    with torch.no_grad():
        probs_s_open, probs_t_open, dyn_score, _ = meta_loop_model(bx, apply_intervention=False)

        # Record Sys2 cognitive trigger trace
        entropy_s = -torch.sum(probs_s_open * torch.log(probs_s_open + 1e-9), dim=-1)
        emb_norm = F.normalize(meta_loop_model.model(bx, return_emb=True)[2], p=2, dim=1)
        cos_sim = torch.matmul(emb_norm, meta_loop_model.prototypes.T)
        proto_dist = 1.0 - torch.max(cos_sim, dim=1)[0]
        trigger_s = (entropy_s > meta_loop_model.entropy_s_thresh) | (proto_dist > meta_loop_model.proto_dist_thresh)

    probs_s_closed, _, _, _ = meta_loop_model(bx, apply_intervention=True)

    sys1_probs_s_all.append(probs_s_open.cpu().numpy())
    sys2_probs_s_all.append(probs_s_closed.detach().cpu().numpy())
    sys1_probs_t_all.append(probs_t_open.cpu().numpy())
    dynamics_scores_all.append(dyn_score.cpu().numpy())
    sys2_triggers_all.append(trigger_s.cpu().numpy())

sys1_probs_s_all = np.concatenate(sys1_probs_s_all)
sys2_probs_s_all = np.concatenate(sys2_probs_s_all)
sys1_probs_t_all = np.concatenate(sys1_probs_t_all)
dynamics_scores_all = np.concatenate(dynamics_scores_all)
sys2_triggers_all = np.concatenate(sys2_triggers_all)

smooth_sys1_probs_s = moving_avg_probs(sys1_probs_s_all, k=3)
smooth_sys2_probs_s = moving_avg_probs(sys2_probs_s_all, k=3)
smooth_sys1_probs_t = moving_avg_probs(sys1_probs_t_all, k=3)

sys1_preds_s = np.argmax(smooth_sys1_probs_s, axis=1)
sys2_preds_s = np.argmax(smooth_sys2_probs_s, axis=1)
sys1_confs_s = np.max(smooth_sys1_probs_s, axis=1)

print("\n" + "="*50)

# [1. State cognitive system]
print("\n [1. State Cognitive System]")
sys1_acc = (sys1_preds_s == Y_test_fut).mean() * 100
sys2_acc = (sys2_preds_s == Y_test_fut).mean() * 100
sys1_ece = calculate_ece(smooth_sys1_probs_s, Y_test_fut) * 100
sys2_ece = calculate_ece(smooth_sys2_probs_s, Y_test_fut) * 100
print(f"  · Accuracy: Sys1 {sys1_acc:.2f}% | Sys2 {sys2_acc:.2f}%")
print(f"  · ECE: Sys1 {sys1_ece:.2f}% | Sys2 {sys2_ece:.2f}%")

# [2. Transition warning system]
print("\n [2. Transition Warning System]")

smooth_sys1_probs_t_1d = np.array(smooth_sys1_probs_t).flatten()
Y_test_is_trans_1d = np.array(Y_test_is_trans).flatten()

PROB_THRESH = 0.5
stable_triggers_tensor = temporal_filter(
    torch.tensor(smooth_sys1_probs_t_1d, dtype=torch.float32),
    threshold=PROB_THRESH,
    k=3
)
final_trans_preds_1d = apply_cooldown(stable_triggers_tensor.numpy(), cooldown=5)

def calculate_transition_metrics_robust(y_true, y_pred, y_prob, pre_frames=15, post_frames=5, fs=1.92):
    """
    pre_frames=15: early warning up to ~8 sec before transition
    post_frames=5: tolerance for ~2.5 sec phase delay from filtering
    """
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_prob)
    auprc = auc(recall_curve, precision_curve)

    true_trans_indices = np.where(y_true > 0.5)[0]
    pred_trans_indices = np.where(y_pred > 0.5)[0]

    hits = 0
    lead_times = []
    valid_alarms = set()

    for idx in true_trans_indices:
        start_idx = max(0, idx - pre_frames)
        end_idx = min(len(y_pred), idx + post_frames + 1)

        window_preds = y_pred[start_idx:end_idx]

        if np.any(window_preds > 0.5):
            hits += 1
            alarm_indices = np.where(window_preds > 0.5)[0]
            first_alarm_absolute = start_idx + alarm_indices[0]

            lead_time_sec = (idx - first_alarm_absolute) / fs
            lead_times.append(lead_time_sec)

            for a_idx in alarm_indices:
                valid_alarms.add(start_idx + a_idx)

    event_recall = hits / len(true_trans_indices) if len(true_trans_indices) > 0 else 0

    total_alarms = len(pred_trans_indices)
    valid_alarm_count = len(valid_alarms)
    precision_val = valid_alarm_count / total_alarms if total_alarms > 0 else 0

    positive_lead_times = [lt for lt in lead_times if lt > 0]
    avg_lead_time = np.mean(positive_lead_times) if len(positive_lead_times) > 0 else 0

    return auprc, precision_val, event_recall, avg_lead_time

auprc, precision_val, event_recall, avg_lead_time = calculate_transition_metrics_robust(
    Y_test_is_trans_1d, final_trans_preds_1d, smooth_sys1_probs_t_1d
)

print(f"  · AUPRC: {auprc:.4f}")
print(f"  · Precision: {precision_val*100:.2f}%")
print(f"  · Event Recall: {event_recall*100:.2f}%")
print(f"  · Lead Time: {avg_lead_time:.2f} sec")
print(f"  · (Debug): {int(final_trans_preds_1d.sum())}")

# [3. Meta-cognitive intervention]
print("\n [3. Meta-Cognitive Layer]")
uncertain_mask_s = sys1_confs_s < 0.40
print(f"  · Intervention triggered samples (uncertainty < 0.40): {uncertain_mask_s.sum()}")
if uncertain_mask_s.sum() > 0:
    n_fixed = ((sys1_preds_s[uncertain_mask_s] != Y_test_fut[uncertain_mask_s]) &
               (sys2_preds_s[uncertain_mask_s] == Y_test_fut[uncertain_mask_s])).sum()
    n_broken = ((sys1_preds_s[uncertain_mask_s] == Y_test_fut[uncertain_mask_s]) &
                (sys2_preds_s[uncertain_mask_s] != Y_test_fut[uncertain_mask_s])).sum()
    print(f"  · Net correction rate (low confidence): {(n_fixed - n_broken) / uncertain_mask_s.sum() * 100:.2f}% (saved: {n_fixed}, harmed: {n_broken})")
print("="*50)


>>> [6] Dynamics-Driven + Temporal Filter...


Dual-system inference: 100%|██████████| 671/671 [00:14<00:00, 46.94it/s]




 [1. State Cognitive System]
  · Accuracy: Sys1 70.84% | Sys2 79.36%
  · ECE: Sys1 10.13% | Sys2 15.20%

 [2. Transition Warning System]
  · AUPRC: 0.0774
  · Precision: 22.58%
  · Event Recall: 73.46%
  · Lead Time: 6.76 sec
  · (Debug): 13048

 [3. Meta-Cognitive Layer]
  · Intervention triggered samples (uncertainty < 0.40): 9708
  · Net correction rate (low confidence): 31.98% (saved: 3706, harmed: 601)
